In [1]:
import asyncio
import time
import httpx
from urllib.parse import urlencode, quote
from typing import Optional


async def fetch_papers(
    max_results: Optional[int] = 5,
    start: int = 0,
    sort_by: str = "submittedDate",
    sort_order: str = "descending",
    from_date: Optional[str] = None,
    to_date: Optional[str] = None,
):
    # Configuration
    search_category = "cs.AI"
    base_url = "https://export.arxiv.org/api/query"
    timeout_seconds = 30

    # Build search query
    search_query = f"cat:{search_category}"

    if from_date or to_date:
        date_from = f"{from_date}0000" if from_date else "*"
        date_to = f"{to_date}2359" if to_date else "*"
        search_query += f" AND submittedDate:[{date_from}+TO+{date_to}]"

    params = {
        "search_query": search_query,
        "start": start,
        "max_results": min(max_results, 2000),
        "sortBy": sort_by,
        "sortOrder": sort_order,
    }

    safe = ":+[]"
    url = f"{base_url}?{urlencode(params, quote_via=quote, safe=safe)}"

    print("Request URL:")
    print(url)

    # Make request
    async with httpx.AsyncClient(timeout=timeout_seconds) as client:
        response = await client.get(url)
        response.raise_for_status()

    print(f"\nStatus code: {response.status_code}")
    print(f"Response size: {len(response.text)} characters")

    return response.text


# Run the test
xml_response = await fetch_papers(max_results=5)

print("\n--- Response ---")
print(xml_response[:3000])

Request URL:
https://export.arxiv.org/api/query?search_query=cat:cs.AI&start=0&max_results=5&sortBy=submittedDate&sortOrder=descending


ConnectError: [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1010)